# Register the synthetic Fabric POC dataset

In [ ]:
%%configure
{
  "defaultLakehouse": {
    "name": {"variableName": "$(/**/vl_ingestion/lakehouse_name)"},
    "id": {"variableName": "$(/**/vl_ingestion/lakehouse_id)"},
    "workspaceId": {"variableName": "$(/**/vl_ingestion/workspace_id)"}
  }
}

In [ ]:
import json
import notebookutils

library = notebookutils.variableLibrary.getLibrary("vl_ingestion")
if library.getVariable("environment") != "dev":
    raise ValueError("The POC setup runs in DEV")
workspace_id = library.getVariable("workspace_id")
lakehouse_id = library.getVariable("lakehouse_id")
root = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Files/poc_source"
if not notebookutils.fs.mkdirs("Files/poc_source/orders"):
    raise OSError("Could not prepare the synthetic source folder")
options = json.dumps({"root": root})
spark.createDataFrame([("poc_onelake", "dev", "adls", options)],
    "connection_id string, environment string, connector string, options_json string"
).createOrReplaceTempView("poc_connection")
spark.createDataFrame([("poc__orders","poc_onelake","file_incremental",True,"poc",
                       json.dumps({"prefix":"orders","glob":"*.csv"}))],
    "dataset_id string, connection_id string, load_mode string, enabled boolean, run_group string, options_json string"
).createOrReplaceTempView("poc_dataset")
spark.sql("""MERGE INTO ops.connection t USING poc_connection s
    ON t.connection_id = s.connection_id AND t.environment = s.environment
    WHEN NOT MATCHED THEN INSERT *""")
spark.sql("""MERGE INTO ops.dataset t USING poc_dataset s
    ON t.dataset_id = s.dataset_id WHEN NOT MATCHED THEN INSERT *""")
for statement in "CREATE TABLE IF NOT EXISTS ops.quality_rule (\n  dataset_id STRING, rule_id STRING, rule_type STRING, column_name STRING,\n  threshold BIGINT, severity STRING, enabled BOOLEAN\n) USING DELTA;\n\nCREATE TABLE IF NOT EXISTS ops.quality_result (\n  dataset_id STRING, batch_id STRING, landing_attempt_id STRING, evaluation_id STRING,\n  rule_id STRING, measured BIGINT, threshold BIGINT, severity STRING,\n  status STRING, recorded_utc TIMESTAMP\n) USING DELTA;\n\nMERGE INTO ops.quality_rule target\nUSING (\n  SELECT * FROM VALUES\n    ('poc__orders', 'rows_expected', 'min_rows', '', 1L, 'fail', true),\n    ('poc__orders', 'id_required', 'null_or_blank', 'order_id', 0L, 'fail', true),\n    ('poc__orders', 'id_unique', 'duplicate_keys', 'order_id', 0L, 'fail', true),\n    ('poc__orders', 'date_valid', 'invalid_date', 'order_date', 0L, 'fail', true),\n    ('poc__orders', 'amount_valid', 'invalid_long', 'amount_minor', 0L, 'fail', true)\n  AS rules(dataset_id, rule_id, rule_type, column_name, threshold, severity, enabled)\n) seed\nON target.dataset_id = seed.dataset_id AND target.rule_id = seed.rule_id\nWHEN NOT MATCHED THEN INSERT *;\n".split(';'):
    if statement.strip():
        spark.sql(statement)